In [1]:
import ee 
from RadGEEToolbox import GenericCollection, LandsatCollection, get_palette
# import GEE_UBM
from GEE_UBM import InputCollections, build_model_ready_collection, OriginalUBMRun, ModifiedUBM1Run, check_merged_collection
import geemap
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd
import os

In [2]:
service_account = 'localpythonscripts@ut-gee-ugs-bsf-dev.iam.gserviceaccount.com'
credentials = ee.ServiceAccountCredentials(service_account, 'C:\\Users\\mradwin\\ut-gee-ugs-bsf-dev-53dcc5d729e0.json')
ee.Initialize(credentials=credentials)

In [3]:
UT_boundary = ee.FeatureCollection("projects/ut-gee-ugs-bsf-dev/assets/Utah_Regional_Boundary").geometry()
dirty_devil_roi = ee.FeatureCollection('projects/ut-gee-ugs-bsf-dev/assets/dirty_devil_gage_drainage_area').geometry()
duschesne_roi = ee.FeatureCollection('projects/ut-gee-ugs-bsf-dev/assets/duschesne_gage_drainage_area').geometry()
price_roi = ee.FeatureCollection('projects/ut-gee-ugs-bsf-dev/assets/price_river_gage_drainage_area').geometry()
sevier_roi = ee.FeatureCollection('projects/ut-gee-ugs-bsf-dev/assets/sevier_river_drainage_area').geometry()
uinta_roi = ee.FeatureCollection('projects/ut-gee-ugs-bsf-dev/assets/uinta_river_drainage_area').geometry()
weber_roi = ee.FeatureCollection('projects/ut-gee-ugs-bsf-dev/assets/weber_river_oakley_drainage_area').geometry()

Using the DEM to calculate TPI and ruggedness rasters

In [4]:
dem = ee.Image('USGS/SRTMGL1_003') #.clip(UT_boundary).rename('elevation')
dem_proj = dem.projection() #.mean()
dem = dem.multiply(3.28084).clip(UT_boundary).setDefaultProjection(dem_proj).rename('elevation')
# dem = dem.multiply(3.28084).clip(UT_boundary).reproject(crs='EPSG:32612', scale=30).rename('elevation')
# dem = dem.clip(UT_boundary).reproject(crs='EPSG:32612', scale=30).rename('elevation')


slope = ee.Terrain.slope(dem).setDefaultProjection(dem_proj)
# print(dem.projection().nominalScale().getInfo())

aspect = ee.Terrain.aspect(dem).setDefaultProjection(dem_proj)
# print(dem.projection().nominalScale().getInfo())

# Calculate TPI (Topographic Position Index)
# (Subtracts pixel elevation from the average of its neighbors)
# A large neighborhood (e.g., 20 pixels) helps identify valleys vs ridges.
# tpi = dem.subtract(dem.focal_mean(20, 'circle', 'pixels')).rename('TPI').setDefaultProjection(dem_proj)
# tpi = dem.subtract(dem.focal_mean(20, 'circle', 'pixels')).rename('TPI') #.setDefaultProjection(crs='EPSG:32612', scale=30)
tpi = ee.Image('projects/ut-gee-ugs-bsf-dev/assets/Utah_TPI_SRTM_30m')
print(tpi.projection().nominalScale().getInfo())

ruggedness = tpi.abs().rename('Ruggedness').setDefaultProjection(dem_proj)
lithology = ee.Image('projects/ut-gee-ugs-bsf-dev/assets/Utah_Lithology_Rasters/Utah_Lithology_USGS_RockTypes').clip(UT_boundary).rename('lithology')
land_cover = ee.ImageCollection('USGS/NLCD_RELEASES/2021_REL/NLCD').first()\
                                        .select('landcover').clip(UT_boundary)
geomaterials = ee.FeatureCollection('projects/ut-gee-ugs-bsf-dev/assets/GeoMaterial_USGS_CNGM_Utah')

30


In [5]:
lithologies = geomaterials.aggregate_array('GeoMateria').distinct().getInfo()
print(lithologies)

['Coarse-grained, mafic-composition intrusive igneous rock', 'Alluvial sediment', 'Peat and muck', 'Playa sediment', 'Sedimentary material', 'Medium and high-grade regional metamorphic rock, of unspecified origin', 'Sedimentary rock', 'Metamorphic rock', 'Clastic sedimentary rock', 'Mostly sandstone', 'Clastic sediment', 'Debris flows, landslides, and other localized mass-movement sediment', 'Eolian sediment', 'Extrusive igneous material', 'Felsic-composition pyroclastic flows', 'Glacial till', 'Coarse-grained intrusive igneous rock', 'Coarse-grained, felsic-composition intrusive igneous rock', 'Intermediate-composition lava flows', 'Intrusive igneous rock', 'Lacustrine sediment', 'Mafic-composition lava flows', 'Metasedimentary rock', 'Sandstone and mudstone', 'Mostly carbonate rock', 'Sedimentary and extrusive igneous material', 'Water or ice']


In [6]:
lith_list = ['Coarse-grained, mafic-composition intrusive igneous rock', 'Alluvial sediment', 'Peat and muck', 
             'Playa sediment', 'Sedimentary material', 'Medium and high-grade regional metamorphic rock, of unspecified origin', 
             'Sedimentary rock', 'Metamorphic rock', 'Clastic sedimentary rock', 'Mostly sandstone', 'Clastic sediment', 
             'Debris flows, landslides, and other localized mass-movement sediment', 'Eolian sediment', 'Extrusive igneous material', 
             'Felsic-composition pyroclastic flows', 'Glacial till', 'Coarse-grained intrusive igneous rock', 
             'Coarse-grained, felsic-composition intrusive igneous rock', 'Intermediate-composition lava flows', 'Intrusive igneous rock', 
             'Lacustrine sediment', 'Mafic-composition lava flows', 'Metasedimentary rock', 'Sandstone and mudstone', 'Mostly carbonate rock', 
             'Sedimentary and extrusive igneous material', 'Water or ice']

sedimentary_ids = ['Peat and muck', 'Playa sediment', 'Sedimentary material','Sedimentary rock','Clastic sedimentary rock', 'Mostly sandstone', 'Clastic sediment', 
                    'Eolian sediment', 'Lacustrine sediment', 'Sandstone and mudstone', 'Mostly carbonate rock']
alluvial_ids = ['Alluvial sediment', 'Debris flows, landslides, and other localized mass-movement sediment', 'Glacial till']
crystalline_ids = ['Coarse-grained, mafic-composition intrusive igneous rock', 'Medium and high-grade regional metamorphic rock, of unspecified origin', 'Metamorphic rock',
                   'Extrusive igneous material', 'Felsic-composition pyroclastic flows', 'Coarse-grained intrusive igneous rock', 
                    'Coarse-grained, felsic-composition intrusive igneous rock', 'Intermediate-composition lava flows', 'Intrusive igneous rock',
                    'Mafic-composition lava flows', 'Metasedimentary rock',  'Sedimentary and extrusive igneous material']
other_ids = ['Water or ice']

soil_scalar_dict = {}

for name in sedimentary_ids:
    soil_scalar_dict[name] = 5.0
for name in alluvial_ids:
    soil_scalar_dict[name] = 3.0
for name in crystalline_ids:
    soil_scalar_dict[name] = 2.0
for name in other_ids:
    soil_scalar_dict[name] = 1.0

print(soil_scalar_dict)

{'Peat and muck': 5.0, 'Playa sediment': 5.0, 'Sedimentary material': 5.0, 'Sedimentary rock': 5.0, 'Clastic sedimentary rock': 5.0, 'Mostly sandstone': 5.0, 'Clastic sediment': 5.0, 'Eolian sediment': 5.0, 'Lacustrine sediment': 5.0, 'Sandstone and mudstone': 5.0, 'Mostly carbonate rock': 5.0, 'Alluvial sediment': 3.0, 'Debris flows, landslides, and other localized mass-movement sediment': 3.0, 'Glacial till': 3.0, 'Coarse-grained, mafic-composition intrusive igneous rock': 2.0, 'Medium and high-grade regional metamorphic rock, of unspecified origin': 2.0, 'Metamorphic rock': 2.0, 'Extrusive igneous material': 2.0, 'Felsic-composition pyroclastic flows': 2.0, 'Coarse-grained intrusive igneous rock': 2.0, 'Coarse-grained, felsic-composition intrusive igneous rock': 2.0, 'Intermediate-composition lava flows': 2.0, 'Intrusive igneous rock': 2.0, 'Mafic-composition lava flows': 2.0, 'Metasedimentary rock': 2.0, 'Sedimentary and extrusive igneous material': 2.0, 'Water or ice': 1.0}


History of runs in collapsed cell below. Everything past v30 utilized the new version of the model and the fixed precipitation dataset

In [ ]:
#V20 Scalar Dictionary 
# soil_scalar_dict = {'Peat and muck': 5.0, 
#                     'Playa sediment': 5.0, 
#                     'Sedimentary material': 5.0, 
#                     'Sedimentary rock': 5.0, 
#                     'Clastic sedimentary rock': 5.0, 
#                     'Mostly sandstone': 5.0, 
#                     'Clastic sediment': 5.0, 
#                     'Eolian sediment': 5.0, 
#                     'Lacustrine sediment': 5.0, 
#                     'Sandstone and mudstone': 5.0, 
#                     'Mostly carbonate rock': 5.0, 

#                     'Alluvial sediment': 3.0, 
#                     'Debris flows, landslides, and other localized mass-movement sediment': 3.0, 
#                     'Glacial till': 3.0, 

#                     'Coarse-grained, mafic-composition intrusive igneous rock': 2.0, 
#                     'Medium and high-grade regional metamorphic rock, of unspecified origin': 2.0, 
#                     'Metamorphic rock': 2.0, 
#                     'Extrusive igneous material': 2.0, 
#                     'Felsic-composition pyroclastic flows': 2.0, 
#                     'Coarse-grained intrusive igneous rock': 2.0, 
#                     'Coarse-grained, felsic-composition intrusive igneous rock': 2.0, 
#                     'Intermediate-composition lava flows': 2.0, 
#                     'Intrusive igneous rock': 2.0, 
#                     'Mafic-composition lava flows': 2.0, 
#                     'Metasedimentary rock': 2.0, 
#                     'Sedimentary and extrusive igneous material': 2.0,
                     
#                     'Water or ice': 1.0}

# V21 Scalar Dictionary - The "Deep Regolith" Correction
# soil_scalar_dict = {
#     # --- GROUP A: SEDIMENTARY (The Infinite Sponges) -> 20.0x ---
#     # Targets: Big Creek, West Canyon, Muddy Creek
#     # Correcting for deep fractured bedrock storage missing in gNATSGO.
#     'Sedimentary rock': 20.0, 
#     'Clastic sedimentary rock': 20.0, 
#     'Mostly sandstone': 20.0, 
#     'Sandstone and mudstone': 20.0, 
#     'Peat and muck': 20.0, 
#     'Playa sediment': 20.0, 
#     'Sedimentary material': 20.0, 
#     'Clastic sediment': 20.0, 
#     'Eolian sediment': 20.0, 
#     'Lacustrine sediment': 20.0, 
#     'Mostly carbonate rock': 20.0, 

#     # --- GROUP B: VOLCANIC (The Fractured Lava) -> 10.0x ---
#     # Targets: Mammoth Creek
#     # Lava flows are porous and rubbly. 2.0x was far too tight.
#     'Extrusive igneous material': 10.0, 
#     'Intermediate-composition lava flows': 10.0, 
#     'Mafic-composition lava flows': 10.0, 
#     'Felsic-composition pyroclastic flows': 10.0,
#     'Sedimentary and extrusive igneous material': 10.0,

#     # --- GROUP C: ALLUVIAL & TILL (The Deep Fill) -> 6.0x ---
#     # Targets: Whiterocks, Castle Creek
#     'Glacial till': 6.0, 
#     'Alluvial sediment': 6.0, 
#     'Debris flows, landslides, and other localized mass-movement sediment': 6.0, 

#     # --- GROUP D: CRYSTALLINE (Hard Rock) -> 4.0x ---
#     # Targets: Farmington, Blacks Fork
#     # Even "hard" rock has deep fractures in these mountains.
#     'Coarse-grained, mafic-composition intrusive igneous rock': 4.0, 
#     'Medium and high-grade regional metamorphic rock, of unspecified origin': 4.0, 
#     'Metamorphic rock': 4.0, 
#     'Coarse-grained intrusive igneous rock': 4.0, 
#     'Coarse-grained, felsic-composition intrusive igneous rock': 4.0, 
#     'Intrusive igneous rock': 4.0, 
#     'Metasedimentary rock': 4.0, # Uinta Mountain Group
    
#     # Defaults
#     'Water or ice': 1.0
# }

# # V22 Scalar Dictionary - "The Infinite Sponge" (Corrected)
# soil_scalar_dict = {
#     # --- GROUP A: SEDIMENTARY (The Deep Aquifers) -> 50.0x ---
#     # Targets: Big Creek, West Canyon, Muddy Creek.
#     # We are modeling deep vadose zones that prevent ALL saturation excess.
#     'Sedimentary rock': 50.0, 
#     'Clastic sedimentary rock': 50.0, 
#     'Mostly sandstone': 50.0, 
#     'Sandstone and mudstone': 50.0, 
#     'Peat and muck': 50.0, 
#     'Playa sediment': 50.0, 
#     'Sedimentary material': 50.0, 
#     'Clastic sediment': 50.0, 
#     'Eolian sediment': 50.0, 
#     'Lacustrine sediment': 50.0, 
#     'Mostly carbonate rock': 50.0, 

#     # --- GROUP B: VOLCANIC (The Lava Fields) -> 30.0x ---
#     # Targets: Mammoth Creek.
#     # 30x helps simulate the deep fractured porosity of lava flows.
#     'Extrusive igneous material': 30.0, 
#     'Intermediate-composition lava flows': 30.0, 
#     'Mafic-composition lava flows': 30.0, 
#     'Felsic-composition pyroclastic flows': 30.0,
#     'Sedimentary and extrusive igneous material': 30.0,

#     # --- GROUP C: ALLUVIAL & TILL (Valley Fill) -> 6.0x (LOCKED) ---
#     # Targets: Whiterocks (+73%).
#     # We keep this at 6.0x because Whiterocks is already in the "Goldilocks Zone".
#     # We ignore Castle Creek's artificial errors.
#     'Glacial till': 6.0, 
#     'Alluvial sediment': 6.0, 
#     'Debris flows, landslides, and other localized mass-movement sediment': 6.0, 

#     # --- GROUP D: CRYSTALLINE (Fractured Basement) -> 8.0x ---
#     # Targets: Farmington (+125%), Blacks Fork (+200%)
#     # Doubling the scalar (4x -> 8x) should bring these into the +50-100% range.
#     'Coarse-grained, mafic-composition intrusive igneous rock': 8.0, 
#     'Medium and high-grade regional metamorphic rock, of unspecified origin': 8.0, 
#     'Metamorphic rock': 8.0, 
#     'Coarse-grained intrusive igneous rock': 8.0, 
#     'Coarse-grained, felsic-composition intrusive igneous rock': 8.0, 
#     'Intrusive igneous rock': 8.0, 
#     'Metasedimentary rock': 8.0, # Uinta Mountain Group
    
#     # Defaults
#     'Water or ice': 1.0
# }

# V23 Scalar Dictionary - "The Balanced Attack"
# soil_scalar_dict = {
#     # --- GROUP A: SEDIMENTARY (The Bottomless Pit) -> 100.0x ---
#     # Targets: Big Creek (+950%), Muddy Creek (+250%).
#     # Doubling down to crush the remaining saturation excess.
#     'Sedimentary rock': 100.0, 
#     'Clastic sedimentary rock': 100.0, 
#     'Mostly sandstone': 100.0, 
#     'Sandstone and mudstone': 100.0, 
#     'Peat and muck': 100.0, 
#     'Playa sediment': 100.0, 
#     'Sedimentary material': 100.0, 
#     'Clastic sediment': 100.0, 
#     'Eolian sediment': 100.0, 
#     'Lacustrine sediment': 100.0, 
#     'Mostly carbonate rock': 100.0, 

#     # --- GROUP B: VOLCANIC (The Deep Fractures) -> 60.0x ---
#     # Targets: Mammoth Creek (+280%).
#     # Doubling storage to cut the remaining error.
#     'Extrusive igneous material': 60.0, 
#     'Intermediate-composition lava flows': 60.0, 
#     'Mafic-composition lava flows': 60.0, 
#     'Felsic-composition pyroclastic flows': 60.0,
#     'Sedimentary and extrusive igneous material': 60.0,

#     # --- GROUP C: METASEDIMENTARY (The Compromise) -> 15.0x ---
#     # Targets: Blacks Fork (Needs Help), Whiterocks (Needs Safety).
#     # 15x doubles the storage for Blacks Fork but protects Whiterocks from 
#     # dropping below 0% runoff.
#     'Metasedimentary rock': 15.0,

#     # --- GROUP D: ALLUVIAL & TILL (The Bullseye) -> 6.0x (LOCKED) ---
#     # Targets: Whiterocks (+58%).
#     # Locked to preserve calibration.
#     'Glacial till': 6.0, 
#     'Alluvial sediment': 6.0, 
#     'Debris flows, landslides, and other localized mass-movement sediment': 6.0, 

#     # --- GROUP E: CRYSTALLINE (Hard Rock Correction) -> 5.0x ---
#     # Targets: Farmington Creek (-32%).
#     # V22 (8x) killed the runoff. Backing off to 5x to restore positive bias.
#     'Coarse-grained, mafic-composition intrusive igneous rock': 5.0, 
#     'Medium and high-grade regional metamorphic rock, of unspecified origin': 5.0, 
#     'Metamorphic rock': 5.0, 
#     'Coarse-grained intrusive igneous rock': 5.0, 
#     'Coarse-grained, felsic-composition intrusive igneous rock': 5.0, 
#     'Intrusive igneous rock': 5.0, 
#     'Intrusive igneous rock': 5.0, 
    
#     # Defaults
#     'Water or ice': 1.0
# }

# # V24 Scalar Dictionary - "The Split & Balance"
# soil_scalar_dict = {
#     # --- GROUP A: GENERIC SEDIMENTARY (The Deep Sponge) -> 200.0x ---
#     # Targets: Big Creek (+414%).
#     # 100x wasn't enough. Going massive to stop the saturation excess explosion.
#     # Note: This will also apply to West Canyon (which is dry/karst), keeping it dry. Good.
#     'Sedimentary rock': 200.0, 
#     'Peat and muck': 200.0, 
#     'Playa sediment': 200.0, 
#     'Sedimentary material': 200.0, 
#     'Eolian sediment': 200.0, 
#     'Lacustrine sediment': 200.0, 
#     'Mostly carbonate rock': 200.0, 

#     # --- GROUP B: SPECIFIC SEDIMENTARY (The Rescue) -> 40.0x ---
#     # Targets: Muddy Creek (-100%).
#     # 100x killed it. Backing off to resurrect runoff.
#     'Clastic sedimentary rock': 40.0, 
#     'Mostly sandstone': 40.0, 
#     'Sandstone and mudstone': 40.0, 
#     'Clastic sediment': 40.0, 

#     # --- GROUP C: VOLCANIC (The Locked Win) -> 60.0x ---
#     # Targets: Mammoth Creek (+69%).
#     # PERFECT. Do not touch.
#     'Extrusive igneous material': 60.0, 
#     'Intermediate-composition lava flows': 60.0, 
#     'Mafic-composition lava flows': 60.0, 
#     'Felsic-composition pyroclastic flows': 60.0,
#     'Sedimentary and extrusive igneous material': 60.0,

#     # --- GROUP D: METASEDIMENTARY (The Aggressive Push) -> 25.0x ---
#     # Targets: Blacks Fork (+156%).
#     # Needs more storage to come down.
#     # Warning: This puts downward pressure on Whiterocks.
#     'Metasedimentary rock': 25.0,

#     # --- GROUP E: ALLUVIAL & TILL (The Counter-Balance) -> 4.0x ---
#     # Targets: Whiterocks (+34%).
#     # Lowering from 6x -> 4x to INCREASE runoff, counteracting the Metased boost.
#     'Glacial till': 4.0, 
#     'Alluvial sediment': 4.0, 
#     'Debris flows, landslides, and other localized mass-movement sediment': 4.0, 

#     # --- GROUP F: CRYSTALLINE (The Fine Tune) -> 6.0x ---
#     # Targets: Farmington Creek (+93%).
#     # 5x was too wet, 8x was too dry. 6x is the sweet spot.
#     'Coarse-grained, mafic-composition intrusive igneous rock': 6.0, 
#     'Medium and high-grade regional metamorphic rock, of unspecified origin': 6.0, 
#     'Metamorphic rock': 6.0, 
#     'Coarse-grained intrusive igneous rock': 6.0, 
#     'Coarse-grained, felsic-composition intrusive igneous rock': 6.0, 
#     'Intrusive igneous rock': 6.0, 
    
#     # Defaults
#     'Water or ice': 1.0
# }

# V25 Scalar Dictionary - "The Recharge Correction"
# Context: Global K (Conductivity) has been multiplied by 0.5x.
# This naturally increases Runoff, so we deepen the soils in wet basins to compensate.

# soil_scalar_dict = {
#     # --- GROUP A: GENERIC SEDIMENTARY (The Deep Sponge) -> 300.0x ---
#     # Targets: Big Creek (+303% Runoff).
#     # Increased from 200x to 300x to absorb the extra water from the slower K drain.
#     'Sedimentary rock': 300.0, 
#     'Peat and muck': 300.0, 
#     'Playa sediment': 300.0, 
#     'Sedimentary material': 300.0, 
#     'Eolian sediment': 300.0, 
#     'Lacustrine sediment': 300.0, 
#     'Mostly carbonate rock': 300.0, 

#     # --- GROUP B: SPECIFIC SEDIMENTARY (The Rescue) -> 20.0x ---
#     # Targets: Muddy Creek (-100% Runoff, +87% Recharge).
#     # Lowered from 40x to 20x. Combined with K=0.5x, this MUST spill.
#     'Clastic sedimentary rock': 20.0, 
#     'Mostly sandstone': 20.0, 
#     'Sandstone and mudstone': 20.0, 
#     'Clastic sediment': 20.0, 

#     # --- GROUP C: VOLCANIC (Stable) -> 60.0x ---
#     # Targets: Mammoth Creek (+69% Runoff, +119% Recharge).
#     # K=0.5x will fix the Recharge spike. Runoff might rise slightly (+80%), which is fine.
#     'Extrusive igneous material': 60.0, 
#     'Intermediate-composition lava flows': 60.0, 
#     'Mafic-composition lava flows': 60.0, 
#     'Felsic-composition pyroclastic flows': 60.0,
#     'Sedimentary and extrusive igneous material': 60.0,

#     # --- GROUP D: METASEDIMENTARY (The Final Push) -> 40.0x ---
#     # Targets: Blacks Fork (+105% Runoff).
#     # Increased from 25x to 40x to bring runoff down into the <100% range.
#     'Metasedimentary rock': 40.0,

#     # --- GROUP E: ALLUVIAL & TILL (Stable) -> 4.0x ---
#     # Targets: Whiterocks (+60% Runoff).
#     # K=0.5x might push runoff to +70%. This is safe. Keep 4x.
#     'Glacial till': 4.0, 
#     'Alluvial sediment': 4.0, 
#     'Debris flows, landslides, and other localized mass-movement sediment': 4.0, 

#     # --- GROUP F: CRYSTALLINE (Stable) -> 6.0x ---
#     # Targets: Farmington Creek (+42% Runoff).
#     # K=0.5x will likely push runoff to +50-55%. Perfect.
#     'Coarse-grained, mafic-composition intrusive igneous rock': 6.0, 
#     'Medium and high-grade regional metamorphic rock, of unspecified origin': 6.0, 
#     'Metamorphic rock': 6.0, 
#     'Coarse-grained intrusive igneous rock': 6.0, 
#     'Coarse-grained, felsic-composition intrusive igneous rock': 6.0, 
#     'Intrusive igneous rock': 6.0, 
    
#     # Defaults
#     'Water or ice': 1.0
# }

# V26 Scalar Dictionary - "The Bucket Sizing"
# Context: Global K = 0.5 (LOCKED).
# Goal: Increase soil storage to contain the Runoff spillover from V25.
# Expected Outcome: Runoff drops drastically; Recharge rises slightly (towards +15%).

# soil_scalar_dict = {
#     # --- GROUP A: GENERIC SEDIMENTARY (The Grand Canyon) -> 600.0x ---
#     # Targets: Big Creek (Currently +481% Runoff)
#     # Doubling from 300x to 600x to capture the massive saturation excess.
#     'Sedimentary rock': 600.0, 
#     'Peat and muck': 600.0, 
#     'Playa sediment': 600.0, 
#     'Sedimentary material': 600.0, 
#     'Eolian sediment': 600.0, 
#     'Lacustrine sediment': 600.0, 
#     'Mostly carbonate rock': 600.0, 

#     # --- GROUP B: SPECIFIC SEDIMENTARY (The Taming) -> 60.0x ---
#     # Targets: Muddy Creek (Currently +321% Runoff)
#     # V25 (20x) woke it up, but it's too flashy. Tripling to 60x to dampen the peaks.
#     'Clastic sedimentary rock': 60.0, 
#     'Mostly sandstone': 60.0, 
#     'Sandstone and mudstone': 60.0, 
#     'Clastic sediment': 60.0, 

#     # --- GROUP C: VOLCANIC (The Anchor) -> 60.0x (LOCKED) ---
#     # Targets: Mammoth Creek (Currently +72% Runoff - PERFECT)
#     # Do not touch.
#     'Extrusive igneous material': 60.0, 
#     'Intermediate-composition lava flows': 60.0, 
#     'Mafic-composition lava flows': 60.0, 
#     'Felsic-composition pyroclastic flows': 60.0,
#     'Sedimentary and extrusive igneous material': 60.0,

#     # --- GROUP D: METASEDIMENTARY (The Correction) -> 80.0x ---
#     # Targets: Blacks Fork (Currently +229% Runoff)
#     # Doubling from 40x to 80x.
#     'Metasedimentary rock': 80.0,

#     # --- GROUP E: ALLUVIAL & TILL (The Correction) -> 10.0x ---
#     # Targets: Whiterocks (Currently +236% Runoff)
#     # Increasing from 4x to 10x (~2.5x increase).
#     'Glacial till': 10.0, 
#     'Alluvial sediment': 10.0, 
#     'Debris flows, landslides, and other localized mass-movement sediment': 10.0, 

#     # --- GROUP F: CRYSTALLINE (The Correction) -> 15.0x ---
#     # Targets: Farmington Creek (Currently +260% Runoff)
#     # Increasing from 6x to 15x (~2.5x increase).
#     'Coarse-grained, mafic-composition intrusive igneous rock': 15.0, 
#     'Medium and high-grade regional metamorphic rock, of unspecified origin': 15.0, 
#     'Metamorphic rock': 15.0, 
#     'Coarse-grained intrusive igneous rock': 15.0, 
#     'Coarse-grained, felsic-composition intrusive igneous rock': 15.0, 
#     'Intrusive igneous rock': 15.0, 
    
#     # Defaults
#     'Water or ice': 1.0
# }

# V27 Scalar Dictionary - "The Pulse Restoration"
# Context: Global K = 0.65 (Opened significantly to restore variability).
# Goal: Recharge ~+35% (Pulse restored). Runoff ~+100% (Acceptable limit).
# Strategy: Smaller buckets + Higher K = More Dynamic System.

# soil_scalar_dict = {
#     # --- GROUP A: SEDIMENTARY (The Dynamic Sponge) -> 400.0x ---
#     # Target: Big Creek (+100% Runoff Target).
#     # Reduced from V26 (600x) to restore variability.
#     # Increased from V24 (200x) to control the V25 runoff spike (+481%).
#     'Sedimentary rock': 400.0, 
#     'Peat and muck': 400.0, 
#     'Playa sediment': 400.0, 
#     'Sedimentary material': 400.0, 
#     'Eolian sediment': 400.0, 
#     'Lacustrine sediment': 400.0, 
#     'Mostly carbonate rock': 400.0, 

#     # --- GROUP B: SPECIFIC SEDIMENTARY (The Flow) -> 35.0x ---
#     # Target: Muddy Creek.
#     # V26 (60x) was dry. V25 (20x) was flashy.
#     # 35x is the calculated midpoint to allow flow without explosion.
#     'Clastic sedimentary rock': 35.0, 
#     'Mostly sandstone': 35.0, 
#     'Sandstone and mudstone': 35.0, 
#     'Clastic sediment': 35.0, 

#     # --- GROUP C: VOLCANIC (The Anchor) -> 60.0x ---
#     # Target: Mammoth Creek.
#     # Performance is calibrated. Locking this variable.
#     'Extrusive igneous material': 60.0, 
#     'Intermediate-composition lava flows': 60.0, 
#     'Mafic-composition lava flows': 60.0, 
#     'Felsic-composition pyroclastic flows': 60.0,
#     'Sedimentary and extrusive igneous material': 60.0,

#     # --- GROUP D: METASEDIMENTARY (The Tamer) -> 100.0x ---
#     # Target: Blacks Fork.
#     # Increasing to 100x to bring runoff down from +170%.
#     'Metasedimentary rock': 100.0,

#     # --- GROUP E: ALLUVIAL & TILL (The Pulse) -> 12.0x ---
#     # Target: Whiterocks.
#     # Using 12x (instead of 20x) to ensure the glacial till remains responsive.
#     'Glacial till': 12.0, 
#     'Alluvial sediment': 12.0, 
#     'Debris flows, landslides, and other localized mass-movement sediment': 12.0, 

#     # --- GROUP F: CRYSTALLINE (The Control) -> 40.0x ---
#     # Target: Farmington Creek.
#     # Increasing to 40x to control the +260% spike, but keeping it 
#     # slightly smaller than the 50x safety calculation to help variability.
#     'Coarse-grained, mafic-composition intrusive igneous rock': 40.0, 
#     'Medium and high-grade regional metamorphic rock, of unspecified origin': 40.0, 
#     'Metamorphic rock': 40.0, 
#     'Coarse-grained intrusive igneous rock': 40.0, 
#     'Coarse-grained, felsic-composition intrusive igneous rock': 40.0, 
#     'Intrusive igneous rock': 40.0, 
    
#     # Defaults
#     'Water or ice': 1.0
# }

# V28 !!!
# soil_scalar_dict = {
#     # --- GROUP A: SEDIMENTARY (The Dynamic Balance) -> 300.0x ---
#     # Target: Big Creek.
#     # V27 (400x) was dry (-100%). V25 (300x, K=0.5) was wet (+480%).
#     # V28 (300x, K=1.0) uses the faster drain to control the V25 spill.
#     'Sedimentary rock': 300.0, 
#     'Peat and muck': 300.0, 
#     'Playa sediment': 300.0, 
#     'Sedimentary material': 300.0, 
#     'Eolian sediment': 300.0, 
#     'Lacustrine sediment': 300.0, 
#     'Mostly carbonate rock': 300.0, 

#     # --- GROUP B: CLASTIC (The Pulse) -> 25.0x ---
#     # Target: Muddy Creek.
#     # Decreasing from 35x to compensate for the higher K (1.0).
#     'Clastic sedimentary rock': 25.0, 
#     'Mostly sandstone': 25.0, 
#     'Sandstone and mudstone': 25.0, 
#     'Clastic sediment': 25.0, 

#     # --- GROUP C: VOLCANIC (The Anchor) -> 50.0x ---
#     # Target: Mammoth Creek.
#     # Slight reduction from 60x to maintain runoff against the higher K.
#     'Extrusive igneous material': 50.0, 
#     'Intermediate-composition lava flows': 50.0, 
#     'Mafic-composition lava flows': 50.0, 
#     'Felsic-composition pyroclastic flows': 50.0,
#     'Sedimentary and extrusive igneous material': 50.0,

#     # --- GROUP D: METASEDIMENTARY (The Correction) -> 75.0x ---
#     # Target: Blacks Fork.
#     # Reduced from 100x. The higher K will handle the volume control.
#     'Metasedimentary rock': 75.0,

#     # --- GROUP E: ALLUVIAL & TILL (The Response) -> 10.0x ---
#     # Target: Whiterocks.
#     'Glacial till': 10.0, 
#     'Alluvial sediment': 10.0, 
#     'Debris flows, landslides, and other localized mass-movement sediment': 10.0, 

#     # --- GROUP F: CRYSTALLINE (The Rescue) -> 20.0x ---
#     # Target: Farmington Creek.
#     # V27 (40x) killed all runoff. 20x restores it.
#     # K=1.0 prevents the +260% spike seen in V26.
#     'Coarse-grained, mafic-composition intrusive igneous rock': 20.0, 
#     'Medium and high-grade regional metamorphic rock, of unspecified origin': 20.0, 
#     'Metamorphic rock': 20.0, 
#     'Coarse-grained intrusive igneous rock': 20.0, 
#     'Coarse-grained, felsic-composition intrusive igneous rock': 20.0, 
#     'Intrusive igneous rock': 20.0, 
    
#     # Defaults
#     'Water or ice': 1.0
# }



# basline
# soil_scalar_dict = {
#     'Sedimentary rock': 1.0, 
#     'Peat and muck': 1.0, 
#     'Playa sediment': 1.0, 
#     'Sedimentary material': 1.0, 
#     'Eolian sediment': 10., 
#     'Lacustrine sediment': 1.0, 
#     'Mostly carbonate rock': 1.0, 

#     'Clastic sedimentary rock': 1.0, 
#     'Mostly sandstone': 1.0, 
#     'Sandstone and mudstone': 1.0, 
#     'Clastic sediment': 1.0, 

#     'Extrusive igneous material': 1.0, 
#     'Intermediate-composition lava flows': 1.0, 
#     'Mafic-composition lava flows': 1.0, 
#     'Felsic-composition pyroclastic flows': 1.0,
#     'Sedimentary and extrusive igneous material': 1.0,

#     'Metasedimentary rock': 1.0,

#     'Glacial till': 1.0, 
#     'Alluvial sediment': 1.0, 
#     'Debris flows, landslides, and other localized mass-movement sediment': 1.0, 

#     'Coarse-grained, mafic-composition intrusive igneous rock': 1.0, 
#     'Medium and high-grade regional metamorphic rock, of unspecified origin': 1.0, 
#     'Metamorphic rock': 1.0, 
#     'Coarse-grained intrusive igneous rock': 1.0, 
#     'Coarse-grained, felsic-composition intrusive igneous rock': 1.0, 
#     'Intrusive igneous rock': 1.0, 
    
#     'Water or ice': 1.0
# }

# # v32
# soil_scalar_dict = {
#     # --- SEDIMENTARY GROUP (Target ~0.46 for Big Creek) ---
#     'Sedimentary rock': 0.46, 
#     'Clastic sedimentary rock': 0.44, # Dominant in Muddy Creek
#     'Sandstone and mudstone': 0.15,   # Low scalar needed to balance Muddy Creek (Target 0.28)
#     'Mostly sandstone': 0.30,         # Interpolated
#     'Mostly carbonate rock': 0.20,    # Minor component
#     'Sedimentary material': 0.46,
#     'Playa sediment': 0.46,
#     'Peat and muck': 1.0,             # Keep high (organic)

#     # --- UNCONSOLIDATED GROUP (Alluvial vs Glacial) ---
#     'Alluvial sediment': 0.80,        # High scalar needed for West Canyon/Weber valleys
#     'Clastic sediment': 0.50,         # Intermediate
#     'Glacial till': 0.18,             # LOW scalar needed for Whiterocks/Blacks Fork (Targets ~0.18-0.22)
#     'Debris flows, landslides, and other localized mass-movement sediment': 0.5,
#     'Eolian sediment': 0.5,           # assuming scaling with other changes
#     'Lacustrine sediment': 0.5,

#     # --- IGNEOUS GROUP (Target ~0.32 for Mammoth) ---
#     'Extrusive igneous material': 0.32,
#     'Intermediate-composition lava flows': 0.32,
#     'Mafic-composition lava flows': 0.32,
#     'Felsic-composition pyroclastic flows': 0.50, # Capped (West Canyon artifact suggested >1.0, but 0.5 is safer)
#     'Sedimentary and extrusive igneous material': 0.40, # Mix
#     'Coarse-grained, mafic-composition intrusive igneous rock': 0.32,
#     'Coarse-grained intrusive igneous rock': 0.32,
#     'Coarse-grained, felsic-composition intrusive igneous rock': 0.32,
#     'Intrusive igneous rock': 0.32,

#     # --- METAMORPHIC GROUP ---
#     # Farmington (Metamorphic) target is 0.46 -> Scalar 0.54
#     # Uintas (Metasedimentary) target is ~0.20 -> Scalar 0.20
#     'Medium and high-grade regional metamorphic rock, of unspecified origin': 0.54, 
#     'Metamorphic rock': 0.54,
#     'Metasedimentary rock': 0.20,     # Distinctly lower for Uintas

#     # --- OTHER ---
#     'Water or ice': 0.0               # Should satisfy logic (no soil)
# }

# #v33
# soil_scalar_dict = {
#     # --- FIXING THE UNDER-ESTIMATORS (Farmington, Muddy, Mammoth) ---
#     'Clastic sedimentary rock': 0.20, # Muddy Creek Fix
#     'Medium and high-grade regional metamorphic rock, of unspecified origin': 0.25, # Farmington Fix
#     'Metamorphic rock': 0.25,
#     'Extrusive igneous material': 0.25, # Mammoth Fix
#     'Intermediate-composition lava flows': 0.25,
#     'Mafic-composition lava flows': 0.25,

#     # --- THE COMPROMISE (Big Creek vs West Canyon) ---
#     # Big Creek needs 0.30. West Canyon needs 0.60.
#     # We set 0.40 as the middle ground.
#     'Sedimentary rock': 0.40,         
#     'Sedimentary material': 0.40,
#     'Mostly sandstone': 0.30,         
#     'Mostly carbonate rock': 0.20,    
#     'Sandstone and mudstone': 0.15,   
#     'Playa sediment': 0.46,

#     # --- THE SHIELD (Protecting Weber & West Canyon) ---
#     # Increased to 1.2 to buffer the extra runoff from mountains
#     'Alluvial sediment': 1.2,        
#     'Clastic sediment': 0.50,         
#     'Debris flows, landslides, and other localized mass-movement sediment': 0.5,
#     'Eolian sediment': 0.5,           
#     'Lacustrine sediment': 0.5,

#     # --- PRESERVING THE GOOD (Whiterocks) ---
#     'Glacial till': 0.18,             # Untouched
#     'Metasedimentary rock': 0.20,     # Untouched
    
#     # --- OTHERS ---
#     'Peat and muck': 1.0,
#     'Felsic-composition pyroclastic flows': 0.50, 
#     'Sedimentary and extrusive igneous material': 0.30,
#     'Coarse-grained, mafic-composition intrusive igneous rock': 0.32,
#     'Coarse-grained intrusive igneous rock': 0.32,
#     'Coarse-grained, felsic-composition intrusive igneous rock': 0.32,
#     'Intrusive igneous rock': 0.32,
#     'Water or ice': 0.0
# }

# v34
# soil_scalar_dict = {
#     # --- SEDIMENTARY GROUP ---
#     'Sedimentary rock': 0.40,         # KEEP (Perfect for Big Creek)
#     'Sedimentary material': 0.40,     # Match above
#     'Clastic sedimentary rock': 0.34, # ADJUSTED: Midpoint for Muddy Creek
#     'Sandstone and mudstone': 0.15,   # Keep low (helper for Muddy)
#     'Mostly sandstone': 0.30,         
#     'Mostly carbonate rock': 0.20,    
#     'Playa sediment': 0.46,

#     # --- UNCONSOLIDATED GROUP ---
#     # West Canyon needs a massive sponge to counteract the thin Sedimentary rock
#     # Weber (+20%) will also benefit from this buffering
#     'Alluvial sediment': 1.6,         # INCREASED (Was 1.2, originally 0.8)
#     'Clastic sediment': 0.50,         
#     'Glacial till': 0.18,             # KEEP (Perfect for Whiterocks)
#     'Debris flows, landslides, and other localized mass-movement sediment': 0.5,
#     'Eolian sediment': 0.5,           
#     'Lacustrine sediment': 0.5,
#     'Peat and muck': 1.0,

#     # --- IGNEOUS GROUP ---
#     'Extrusive igneous material': 0.29,                 # ADJUSTED: Midpoint for Mammoth
#     'Intermediate-composition lava flows': 0.29,        # Match above
#     'Mafic-composition lava flows': 0.29,               # Match above
#     'Felsic-composition pyroclastic flows': 0.50, 
#     'Sedimentary and extrusive igneous material': 0.35, # Mix
#     'Coarse-grained, mafic-composition intrusive igneous rock': 0.32,
#     'Coarse-grained intrusive igneous rock': 0.32,
#     'Coarse-grained, felsic-composition intrusive igneous rock': 0.32,
#     'Intrusive igneous rock': 0.32,

#     # --- METAMORPHIC GROUP ---
#     'Medium and high-grade regional metamorphic rock, of unspecified origin': 0.42, # ADJUSTED: Midpoint for Farmington
#     'Metamorphic rock': 0.42,         # Match above
#     'Metasedimentary rock': 0.20,     # KEEP (Perfect for Uintas)

#     # --- OTHER ---
#     'Water or ice': 0.0
# }

# v35
# soil_scalar_dict = {
#     # --- SEDIMENTARY GROUP (Big & Muddy Lever) ---
#     'Sedimentary rock': 0.35,         # Nudged Down (was 0.38) -> Target Big Creek +5%
#     'Sedimentary material': 0.35,     
#     'Clastic sedimentary rock': 0.18, # LOWERED (was 0.25) -> Target Muddy Creek +5%
#     'Sandstone and mudstone': 0.12,   # Keep Low
#     'Mostly sandstone': 0.30,         
#     'Mostly carbonate rock': 0.20,    
#     'Playa sediment': 0.46,

#     # --- UNCONSOLIDATED GROUP (Standardized) ---
#     # Reverted to physical limits. West Canyon will remain an outlier.
#     'Alluvial sediment': 1.0,         # super sponge
#     'Felsic-composition pyroclastic flows': 0.5, # super sponge
#     'Clastic sediment': 0.50,         
#     'Glacial till': 0.18,             # KEEP (Whiterocks is perfect)
#     'Debris flows, landslides, and other localized mass-movement sediment': 0.5,
#     'Eolian sediment': 0.5,           
#     'Lacustrine sediment': 0.5,
#     'Peat and muck': 1.0,

#     # --- IGNEOUS GROUP (Mammoth Lever) ---
#     # Performing well (+6%). Locking it in.
#     'Extrusive igneous material': 0.28,                 
#     'Intermediate-composition lava flows': 0.28,        
#     'Mafic-composition lava flows': 0.28,               
#     'Sedimentary and extrusive igneous material': 0.35, 
#     'Coarse-grained, mafic-composition intrusive igneous rock': 0.32,
#     'Coarse-grained intrusive igneous rock': 0.32,
#     'Coarse-grained, felsic-composition intrusive igneous rock': 0.32,
#     'Intrusive igneous rock': 0.32,

#     # --- METAMORPHIC GROUP (Farmington Lever) ---
#     # Performing well (+4.5%). Locking it in.
#     'Medium and high-grade regional metamorphic rock, of unspecified origin': 0.35, 
#     'Metamorphic rock': 0.35,         
#     'Metasedimentary rock': 0.18,     

#     # --- OTHER ---
#     'Water or ice': 0.0
# }

#v36
# soil_scalar_dict = {
#     # --- SEDIMENTARY GROUP (Big & Muddy Lever) ---
#     'Sedimentary rock': 0.39,         # Calculated for Big Creek +15%
#     'Sedimentary material': 0.39,     
#     'Clastic sedimentary rock': 0.31, # Calculated for Muddy Creek +15%
#     'Sandstone and mudstone': 0.14,   # Interpolated (0.15 was too dry, 0.12 too wet)
#     'Mostly sandstone': 0.30,         
#     'Mostly carbonate rock': 0.20,    
#     'Playa sediment': 0.46,

#     # --- UNCONSOLIDATED GROUP (Standardized) ---
#     'Alluvial sediment': 1.0,         # Capped at physical limit
#     'Felsic-composition pyroclastic flows': 0.5, 
#     'Clastic sediment': 0.50,         
#     'Glacial till': 0.18,             # KEEP (Whiterocks baseline)
#     'Debris flows, landslides, and other localized mass-movement sediment': 0.5,
#     'Eolian sediment': 0.5,           
#     'Lacustrine sediment': 0.5,
#     'Peat and muck': 1.0,

#     # --- IGNEOUS GROUP (Mammoth Lever) ---
#     'Extrusive igneous material': 0.29,                 # Calculated for Mammoth +15%
#     'Intermediate-composition lava flows': 0.29,        
#     'Mafic-composition lava flows': 0.29,               
#     'Sedimentary and extrusive igneous material': 0.35, 
#     'Coarse-grained, mafic-composition intrusive igneous rock': 0.32,
#     'Coarse-grained intrusive igneous rock': 0.32,
#     'Coarse-grained, felsic-composition intrusive igneous rock': 0.32,
#     'Intrusive igneous rock': 0.32,

#     # --- METAMORPHIC GROUP (Farmington Lever) ---
#     'Medium and high-grade regional metamorphic rock, of unspecified origin': 0.42, # Calculated for Farmington +15%
#     'Metamorphic rock': 0.42,         
#     'Metasedimentary rock': 0.19,     # Compromise: Target Whiterocks ~18%, BF ~11%

#     # --- OTHER ---
#     'Water or ice': 0.0
# }

#v37
# soil_scalar_dict = {
#     # --- SEDIMENTARY GROUP (Muddy Lever) ---
#     'Sedimentary rock': 0.39,         # KEEP (Big Creek is perfect)
#     'Sedimentary material': 0.39,     
#     'Clastic sedimentary rock': 0.30, # ADJUSTED (was 0.31) -> Nudge Muddy up
#     'Sandstone and mudstone': 0.13,   # ADJUSTED (was 0.14) -> Nudge Muddy up
#     'Mostly sandstone': 0.30,         
#     'Mostly carbonate rock': 0.20,    
#     'Playa sediment': 0.46,

#     # --- METAMORPHIC GROUP (Farmington Lever) ---
#     'Medium and high-grade regional metamorphic rock, of unspecified origin': 0.40, # ADJUSTED (was 0.42) -> Lift Farmington to 15%
#     'Metamorphic rock': 0.40,         
#     'Metasedimentary rock': 0.19,     # KEEP (Blacks Fork is good)

#     # --- UNCONSOLIDATED GROUP (Weber/Whiterocks Brake) ---
#     'Glacial till': 0.20,             # INCREASED (was 0.18) -> Dries Weber & Whiterocks
#     'Alluvial sediment': 1.0,         
#     'Felsic-composition pyroclastic flows': 0.5, 
#     'Clastic sediment': 0.50,         
#     'Debris flows, landslides, and other localized mass-movement sediment': 0.5,
#     'Eolian sediment': 0.5,           
#     'Lacustrine sediment': 0.5,
#     'Peat and muck': 1.0,

#     # --- IGNEOUS GROUP (Mammoth Lever) ---
#     # Mammoth is 14.7% (Perfect). Do not touch.
#     'Extrusive igneous material': 0.29,                 
#     'Intermediate-composition lava flows': 0.29,        
#     'Mafic-composition lava flows': 0.29,               
#     'Sedimentary and extrusive igneous material': 0.35, 
#     'Coarse-grained, mafic-composition intrusive igneous rock': 0.32,
#     'Coarse-grained intrusive igneous rock': 0.32,
#     'Coarse-grained, felsic-composition intrusive igneous rock': 0.32,
#     'Intrusive igneous rock': 0.32,

#     # --- OTHER ---
#     'Water or ice': 0.0
# }

#v38 - THIS IS A GREAT BALANCE
# soil_scalar_dict = {
#     # --- SEDIMENTARY GROUP (Big Creek Tuning) ---
#     'Sedimentary rock': 0.392,        # MICRO-ADJUST: 0.39 -> 0.392 (Dries Big Creek slightly)
#     'Sedimentary material': 0.392,    
#     'Clastic sedimentary rock': 0.29, # ADJUSTED: 0.30 -> 0.29 (Wets Muddy slightly)
#     'Sandstone and mudstone': 0.13,   
#     'Mostly sandstone': 0.30,         
#     'Mostly carbonate rock': 0.20,    
#     'Playa sediment': 0.46,

#     # --- METAMORPHIC GROUP (Farmington & Uinta Tuning) ---
#     'Medium and high-grade regional metamorphic rock, of unspecified origin': 0.405, # MICRO-ADJUST: 0.40 -> 0.405 (Dries Farmington)
#     'Metamorphic rock': 0.405,        
#     'Metasedimentary rock': 0.18,     # ADJUSTED: 0.19 -> 0.18 (Boosts Blacks Fork/Whiterocks)

#     # --- UNCONSOLIDATED GROUP (Uinta Tuning) ---
#     'Glacial till': 0.19,             # ADJUSTED: 0.20 -> 0.19 (Boosts Whiterocks/Weber)
#     'Alluvial sediment': 1.0,         
#     'Felsic-composition pyroclastic flows': 0.5, 
#     'Clastic sediment': 0.50,         
#     'Debris flows, landslides, and other localized mass-movement sediment': 0.5,
#     'Eolian sediment': 0.5,           
#     'Lacustrine sediment': 0.5,
#     'Peat and muck': 1.0,

#     # --- IGNEOUS GROUP (Mammoth Locked In) ---
#     'Extrusive igneous material': 0.29,                 
#     'Intermediate-composition lava flows': 0.29,        
#     'Mafic-composition lava flows': 0.29,               
#     'Sedimentary and extrusive igneous material': 0.35, 
#     'Coarse-grained, mafic-composition intrusive igneous rock': 0.32,
#     'Coarse-grained intrusive igneous rock': 0.32,
#     'Coarse-grained, felsic-composition intrusive igneous rock': 0.32,
#     'Intrusive igneous rock': 0.32,

#     # --- OTHER ---
#     'Water or ice': 0.0
# }

# v40 - lowering alluvial as a testing step
# soil_scalar_dict = {
#     # --- SEDIMENTARY GROUP (Big Creek Tuning) ---
#     'Sedimentary rock': 0.392,        # MICRO-ADJUST: 0.39 -> 0.392 (Dries Big Creek slightly)
#     'Sedimentary material': 0.392,    
#     'Clastic sedimentary rock': 0.29, # ADJUSTED: 0.30 -> 0.29 (Wets Muddy slightly)
#     'Sandstone and mudstone': 0.13,   
#     'Mostly sandstone': 0.30,         
#     'Mostly carbonate rock': 0.20,    
#     'Playa sediment': 0.46,

#     # --- METAMORPHIC GROUP (Farmington & Uinta Tuning) ---
#     'Medium and high-grade regional metamorphic rock, of unspecified origin': 0.405, # MICRO-ADJUST: 0.40 -> 0.405 (Dries Farmington)
#     'Metamorphic rock': 0.405,        
#     'Metasedimentary rock': 0.18,     # ADJUSTED: 0.19 -> 0.18 (Boosts Blacks Fork/Whiterocks)

#     # --- UNCONSOLIDATED GROUP (Uinta Tuning) ---
#     'Glacial till': 0.19,             # ADJUSTED: 0.20 -> 0.19 (Boosts Whiterocks/Weber)
#     'Alluvial sediment': 0.5,        # 0.8 in v39 
#     'Felsic-composition pyroclastic flows': 0.5, 
#     'Clastic sediment': 0.50,         
#     'Debris flows, landslides, and other localized mass-movement sediment': 0.5,
#     'Eolian sediment': 0.5,           
#     'Lacustrine sediment': 0.5,
#     'Peat and muck': 1.0,

#     # --- IGNEOUS GROUP (Mammoth Locked In) ---
#     'Extrusive igneous material': 0.29,                 
#     'Intermediate-composition lava flows': 0.29,        
#     'Mafic-composition lava flows': 0.29,               
#     'Sedimentary and extrusive igneous material': 0.35, 
#     'Coarse-grained, mafic-composition intrusive igneous rock': 0.32,
#     'Coarse-grained intrusive igneous rock': 0.32,
#     'Coarse-grained, felsic-composition intrusive igneous rock': 0.32,
#     'Intrusive igneous rock': 0.32,

#     # --- OTHER ---
#     'Water or ice': 0.0
# }

# v41
# soil_scalar_dict = {
#     # --- SEDIMENTARY GROUP (Big Creek Tuning) ---
#     'Sedimentary rock': 0.392,        # MICRO-ADJUST: 0.39 -> 0.392 (Dries Big Creek slightly)
#     'Sedimentary material': 0.392,    
#     'Clastic sedimentary rock': 0.35, # ADJUSTED: 0.30 -> 0.29 (Wets Muddy slightly)
#     'Sandstone and mudstone': 0.09,   
#     'Mostly sandstone': 0.30,         
#     'Mostly carbonate rock': 0.20,    
#     'Playa sediment': 0.46,

#     # --- METAMORPHIC GROUP (Farmington & Uinta Tuning) ---
#     'Medium and high-grade regional metamorphic rock, of unspecified origin': 0.405, # MICRO-ADJUST: 0.40 -> 0.405 (Dries Farmington)
#     'Metamorphic rock': 0.405,        
#     'Metasedimentary rock': 0.18,     # ADJUSTED: 0.19 -> 0.18 (Boosts Blacks Fork/Whiterocks)

#     # --- UNCONSOLIDATED GROUP (Uinta Tuning) ---
#     'Glacial till': 0.19,             # ADJUSTED: 0.20 -> 0.19 (Boosts Whiterocks/Weber)
#     'Alluvial sediment': 0.6,        # 0.8 in v39 
#     'Felsic-composition pyroclastic flows': 0.5, 
#     'Clastic sediment': 0.50,         
#     'Debris flows, landslides, and other localized mass-movement sediment': 0.5,
#     'Eolian sediment': 0.5,           
#     'Lacustrine sediment': 0.5,
#     'Peat and muck': 1.0,

#     # --- IGNEOUS GROUP (Mammoth Locked In) ---
#     'Extrusive igneous material': 0.29,                 
#     'Intermediate-composition lava flows': 0.29,        
#     'Mafic-composition lava flows': 0.29,               
#     'Sedimentary and extrusive igneous material': 0.35, 
#     'Coarse-grained, mafic-composition intrusive igneous rock': 0.32,
#     'Coarse-grained intrusive igneous rock': 0.32,
#     'Coarse-grained, felsic-composition intrusive igneous rock': 0.32,
#     'Intrusive igneous rock': 0.32,

#     # --- OTHER ---
#     'Water or ice': 0.0
# }

# v42 - the new reference standard
# soil_scalar_dict = {
#     # --- SEDIMENTARY GROUP (Big Creek Tuning) ---
#     'Sedimentary rock': 0.415,        # 
#     'Sedimentary material': 0.392,    
#     'Clastic sedimentary rock': 0.315, 
#     'Sandstone and mudstone': 0.09,   
#     'Mostly sandstone': 0.30,         
#     'Mostly carbonate rock': 0.20,    
#     'Playa sediment': 0.46,

#     # --- METAMORPHIC GROUP (Farmington & Uinta Tuning) ---
#     'Medium and high-grade regional metamorphic rock, of unspecified origin': 0.405, # MICRO-ADJUST: 0.40 -> 0.405 (Dries Farmington)
#     'Metamorphic rock': 0.405,        
#     'Metasedimentary rock': 0.18,     # ADJUSTED: 0.19 -> 0.18 (Boosts Blacks Fork/Whiterocks)

#     # --- UNCONSOLIDATED GROUP (Uinta Tuning) ---
#     'Glacial till': 0.19,             # ADJUSTED: 0.20 -> 0.19 (Boosts Whiterocks/Weber)
#     'Alluvial sediment': 0.6,        # 0.8 in v39 
#     'Felsic-composition pyroclastic flows': 0.5, 
#     'Clastic sediment': 0.50,         
#     'Debris flows, landslides, and other localized mass-movement sediment': 0.5,
#     'Eolian sediment': 0.5,           
#     'Lacustrine sediment': 0.5,
#     'Peat and muck': 1.0,

#     # --- IGNEOUS GROUP (Mammoth Locked In) ---
#     'Extrusive igneous material': 0.29,                 
#     'Intermediate-composition lava flows': 0.29,        
#     'Mafic-composition lava flows': 0.29,               
#     'Sedimentary and extrusive igneous material': 0.35, 
#     'Coarse-grained, mafic-composition intrusive igneous rock': 0.32,
#     'Coarse-grained intrusive igneous rock': 0.32,
#     'Coarse-grained, felsic-composition intrusive igneous rock': 0.32,
#     'Intrusive igneous rock': 0.32,

#     # --- OTHER ---
#     'Water or ice': 0.0
# }

# # v43
# soil_scalar_dict = {
#     # --- SEDIMENTARY GROUP (Big Creek Tuning) ---
#     'Sedimentary rock': 0.415,        # 
#     'Sedimentary material': 0.392,    
#     'Clastic sedimentary rock': 0.315, 
#     'Sandstone and mudstone': 0.09,   
#     'Mostly sandstone': 0.30,         
#     'Mostly carbonate rock': 0.20,    
#     'Playa sediment': 0.46,

#     # --- METAMORPHIC GROUP (Farmington & Uinta Tuning) ---
#     'Medium and high-grade regional metamorphic rock, of unspecified origin': 0.405, # MICRO-ADJUST: 0.40 -> 0.405 (Dries Farmington)
#     'Metamorphic rock': 0.405,        
#     'Metasedimentary rock': 0.19,     # ADJUSTED: 0.19 -> 0.18 (Boosts Blacks Fork/Whiterocks)

#     # --- UNCONSOLIDATED GROUP (Uinta Tuning) ---
#     'Glacial till': 0.195,             # ADJUSTED: 0.20 -> 0.19 (Boosts Whiterocks/Weber)
#     'Alluvial sediment': 0.6,        # 0.8 in v39 
#     'Felsic-composition pyroclastic flows': 0.5, 
#     'Clastic sediment': 0.50,         
#     'Debris flows, landslides, and other localized mass-movement sediment': 0.5,
#     'Eolian sediment': 0.5,           
#     'Lacustrine sediment': 0.5,
#     'Peat and muck': 1.0,

#     # --- IGNEOUS GROUP (Mammoth Locked In) ---
#     'Extrusive igneous material': 0.31,                 
#     'Intermediate-composition lava flows': 0.29,        
#     'Mafic-composition lava flows': 0.29,               
#     'Sedimentary and extrusive igneous material': 0.35, 
#     'Coarse-grained, mafic-composition intrusive igneous rock': 0.32,
#     'Coarse-grained intrusive igneous rock': 0.32,
#     'Coarse-grained, felsic-composition intrusive igneous rock': 0.32,
#     'Intrusive igneous rock': 0.32,

#     # --- OTHER ---
#     'Water or ice': 0.0
# }

# # v44
# soil_scalar_dict = {
#     # --- SEDIMENTARY GROUP (Big Creek Tuning) ---
#     'Sedimentary rock': 0.415,        # 
#     'Sedimentary material': 0.392,    
#     'Clastic sedimentary rock': 0.315, 
#     'Sandstone and mudstone': 0.09,   
#     'Mostly sandstone': 0.30,         
#     'Mostly carbonate rock': 0.20,    
#     'Playa sediment': 0.46,

#     # --- METAMORPHIC GROUP (Farmington & Uinta Tuning) ---
#     'Medium and high-grade regional metamorphic rock, of unspecified origin': 0.405, # MICRO-ADJUST: 0.40 -> 0.405 (Dries Farmington)
#     'Metamorphic rock': 0.405,        
#     'Metasedimentary rock': 0.19,     # ADJUSTED: 0.19 -> 0.18 (Boosts Blacks Fork/Whiterocks)

#     # --- UNCONSOLIDATED GROUP (Uinta Tuning) ---
#     'Glacial till': 0.19,             # ADJUSTED: 0.20 -> 0.19 (Boosts Whiterocks/Weber)
#     'Alluvial sediment': 0.6,        # 0.8 in v39 
#     'Felsic-composition pyroclastic flows': 0.5, 
#     'Clastic sediment': 0.50,         
#     'Debris flows, landslides, and other localized mass-movement sediment': 0.5,
#     'Eolian sediment': 0.5,           
#     'Lacustrine sediment': 0.5,
#     'Peat and muck': 1.0,

#     # --- IGNEOUS GROUP (Mammoth Locked In) ---
#     'Extrusive igneous material': 0.30,                 
#     'Intermediate-composition lava flows': 0.29,        
#     'Mafic-composition lava flows': 0.29,               
#     'Sedimentary and extrusive igneous material': 0.35, 
#     'Coarse-grained, mafic-composition intrusive igneous rock': 0.32,
#     'Coarse-grained intrusive igneous rock': 0.32,
#     'Coarse-grained, felsic-composition intrusive igneous rock': 0.32,
#     'Intrusive igneous rock': 0.32,

#     # --- OTHER ---
#     'Water or ice': 0.0
# }


In [14]:

# v45 - FINAL ⚠️⚠️⚠️⚠️⚠️⚠️
soil_scalar_dict = {
    # --- SEDIMENTARY GROUP ---
    'Sedimentary rock': 0.415,        
    'Sedimentary material': 0.415,    
    'Clastic sedimentary rock': 0.315, 
    'Clastic sediment': 0.315,
    'Glacial till': 0.19,             
    'Alluvial sediment': 0.6,       
    'Debris flows, landslides, and other localized mass-movement sediment': 0.4,
    'Sandstone and mudstone': 0.09,   
    'Mostly sandstone': 0.415,         
    'Mostly carbonate rock': 0.35,    
    'Playa sediment': 0.20,
    'Eolian sediment': 0.5,           
    'Lacustrine sediment': 0.5,
    'Peat and muck': 0.5,

    # --- METAMORPHIC GROUP ---
    'Medium and high-grade regional metamorphic rock, of unspecified origin': 0.405, 
    'Metamorphic rock': 0.405,        
    'Metasedimentary rock': 0.19,     


    # --- IGNEOUS/VOLCANIC GROUP ---
    'Extrusive igneous material': 0.30,                 
    'Intermediate-composition lava flows': 0.30,        
    'Mafic-composition lava flows': 0.30,               
    'Sedimentary and extrusive igneous material': 0.30, 
    'Coarse-grained, mafic-composition intrusive igneous rock': 0.32,
    'Coarse-grained intrusive igneous rock': 0.32,
    'Coarse-grained, felsic-composition intrusive igneous rock': 0.32,
    'Intrusive igneous rock': 0.32,
    'Felsic-composition pyroclastic flows': 0.30, 

    # --- OTHER ---
    'Water or ice': 0.0
}


def assign_scalar_value(feature):
    # Get the text description
    rock_type = feature.get('GeoMateria')
    
    # Earth Engine Dictionaries allow looking up values by string keys
    # We wrap the python dict into an ee.Dictionary
    gee_scalar_lookup = ee.Dictionary(soil_scalar_dict)
    
    # Get the value (Default to 1.0 if rock type not found)
    scalar_val = gee_scalar_lookup.get(rock_type, 1.0)
    
    # Set the new numeric property
    return feature.set('st_scalar', scalar_val)

# Map over the collection
geomaterials_with_scalar = geomaterials.map(assign_scalar_value)

soil_thickness_raster = ee.Image('projects/ut-gee-ugs-bsf-dev/assets/UT_RandomForest_1km_gNATSGO_soil_thickness_imperviousIncluded_prediction_raster_final')

# scalar_raster = geomaterials_with_scalar.reduceToImage(['st_scalar'], ee.Reducer.first()).setDefaultProjection(crs='EPSG:4326', scale=100).reduceResolution(
#     reducer=ee.Reducer.mean(),
#     maxPixels=1024
# )

scalar_raster = geomaterials_with_scalar.reduceToImage(['st_scalar'], ee.Reducer.first()).reproject(
    crs=soil_thickness_raster.projection(),  
    scale=soil_thickness_raster.projection().nominalScale() 
)

In [15]:
target_proj = ee.Projection('EPSG:32612').atScale(30)

scalar_raster_updated = geomaterials_with_scalar.reduceToImage(['st_scalar'], ee.Reducer.first()).reproject(target_proj)

Export section

In [16]:
# task = ee.batch.Export.image.toAsset(
#     image=scalar_raster,
#     description='Export_Utah_Geomaterials_ST_Scalar_1000m_v45',
#     assetId='projects/ut-gee-ugs-bsf-dev/assets/Utah_USGS_NGMD_Geomaterials_ST_Scalar_1000m',
#     region=UT_boundary,
#     scale=1000,           # Explicitly force 1000m output
#     crs='EPSG:32612',   # Explicitly force UTM Zone 12N
#     maxPixels=1e13      # Allow huge export (Utah is big)
# )

task = ee.batch.Export.image.toAsset(
    image=scalar_raster_updated,
    description='Export_Utah_Geomaterials_ST_Scalar_30m_v45',
    assetId='projects/ut-gee-ugs-bsf-dev/assets/Utah_USGS_NGMD_Geomaterials_ST_Scalar_30m',
    region=UT_boundary,
    scale=30,           # Explicitly force 30m output
    crs='EPSG:32612',   # Explicitly force UTM Zone 12N
    maxPixels=1e13      # Allow huge export (Utah is big)
)

task.start()

In [10]:
# st_scalar_asset = ee.Image('projects/ut-gee-ugs-bsf-dev/assets/Utah_USGS_NGMD_Geomaterials_ST_Scalar_1000m')
st_scalar_asset = ee.Image('projects/ut-gee-ugs-bsf-dev/assets/Utah_USGS_NGMD_Geomaterials_ST_Scalar_800m')

In [11]:
Map = geemap.Map(center=[39.5, -111.5], zoom=7)
# Map.addLayer(dem, {'min': 4000, 'max': 11000, 'palette':get_palette('dem')}, 'Elevation')
# Map.addLayer(slope, {'min': 0, 'max': 60, 'palette':get_palette('jet')}, 'Slope')
# Map.addLayer(slope, {'min': 0, 'max': 60, 'palette':get_palette('inferno')}, 'Slope')
# Map.addLayer(tpi.abs(), {'min': 0, 'max': 50, 'palette':get_palette('inferno')}, 'TPI Abs')
# Map.addLayer(tpi, {'min': -50, 'max': 50, 'palette':get_palette('rdylbu')}, 'TPI')
# Map.addLayer(valley_elev_factor, {'min': 0, 'max': 1, 'palette':get_palette('blues')}, 'Valley Elevation Factor')
# Map.addLayer(valley_flat_factor, {'min': 0, 'max': 1, 'palette':get_palette('blues')}, 'Valley Flat Factor')
# Map.addLayer(valley_intensity, {'min': 0, 'max': 1, 'palette':get_palette('blues')}, 'Valley Intensity')
# Map.addLayer(lithology, {'min': 0, 'max': 12, 'palette':get_palette('jet')}, opacity=0.6, name='Lithology')
# Map.addLayer(final_ksat_scalar_asset, {'min': 0.000005, 'max': 0.005, 'palette':get_palette('greens')}, 'KSat Scalar Asset')
# Map.addLayer(final_ksat_scalar, {'min': 0.000005, 'max': 0.005, 'palette':get_palette('greens')}, 'new KSat Scalar')

# Map.addLayer(valley_intensity_100m, {'min': 0, 'max': 1, 'palette':get_palette('blues')}, 'Valley Intensity')
# Map.addLayer(geok_scalar, {'min': 0.00001, 'max': 1, 'palette':get_palette('blues')}, 'GeoK Scalar')
# Map.addLayer(geok_raster_100m, {'min': 1e-10*m_s_to_m_month_conversion, 'max': 1e-5*m_s_to_m_month_conversion, 'palette': get_palette('inferno')}, 'GeoK Raster 100m')
# Map.addLayer(geok_raster_scaled, {'min': 1e-10*m_s_to_m_month_conversion, 'max': 1e-5*m_s_to_m_month_conversion, 'palette': get_palette('inferno')}, 'GeoK Raster Scaled 100m')
# Map.addLayer(geok_difference, vis_params={'min':-100, 'max':101, 'palette':['red','white','blue']}, name='GeoK Difference')
# Map.addLayer(k_raster_scaled, {
#     'min': 1e-8*m_s_to_m_month_conversion, 
#     'max': 1e-4*m_s_to_m_month_conversion, 
#     'palette': ['blue', 'cyan', 'green', 'yellow', 'red'] # Low K (Blue) -> High K (Red)
# }, 'Hydraulic Conductivity (m/month)')
# Map.addLayer(final_geok_scalar, {'min': 0.00001, 'max': 1, 'palette':get_palette('greens')}, 'new KSat Scalar')
# Map.addLayer(geok_difference, vis_params={'min':-100, 'max':101, 'palette':['red','white','blue']}, name='GeoK Difference')

Map.addLayer(st_scalar_asset, {'min': 0, 'max': 0.5, 'palette': ['brown', 'yellow', 'green', 'cyan']}, 'Soil Thickness Scalar')

# Map.addLayer(dirty_devil_roi, {'color': 'red'}, 'Dirty Devil Gage Drainage Area')
Map.addLayer(duschesne_roi, {'color': 'blue'}, 'Duschesne Gage Drainage Area')
Map.addLayer(price_roi, {'color': 'green'}, 'Price River Gage Drainage Area')
Map.addLayer(sevier_roi, {'color': 'yellow'}, 'Sevier River Gage Drainage Area')
Map.addLayer(uinta_roi, {'color': 'purple'}, 'Uinta River Gage Drainage Area')
Map.addLayer(weber_roi, {'color': 'orange'}, 'Weber River Oakley Gage Drainage Area')


# Map.addLayer(aspect, {'min': 0, 'max': 360}, 'Aspect')
# Map.addLayer(land_cover, {'min': 0, 'max': 95}, 'Land Cover')
# Map.addLayer(crop_types, {'min': 0, 'max': 255}, 'Crop Types')
# Map.addLayer(lithology, {'min': 0, 'max': 12, 'palette':get_palette('jet')}, 'Lithology')
Map

Map(center=[39.5, -111.5], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright'…

In [55]:
col = ee.ImageCollection("projects/sat-io/open-datasets/HiHydroSoilv2_0/ksat")
ksat = col.min().multiply(10).multiply(30.4375).clip(UT_boundary) # convert from cm/day to mm/month
ksat_min = ksat.reduce(ee.Reducer.min())
ksat_max = ksat.reduce(ee.Reducer.max())

rock_palette = [
    "#F0A3FF", "#0075DC", "#993F00", "#4C005C", "#191919", "#005C31", 
    "#2BCE48", "#FFCC99", "#808080", "#94FFB5", "#8F7C00", "#9DCC00", 
    "#C20088", "#003380", "#FFA405", "#FFA8BB", "#426600", "#FF0010", 
    "#5EF1F2", "#00998F", "#E0FF66", "#740AFF", "#990000", "#FFFF80", 
    "#FFFF00", "#FF5005", "#404040"
]
vis_k = {
    'min': -8, # Log10(1e-8)
    'max': -4, # Log10(1e-4)
    'palette': ['blue', 'cyan', 'green', 'yellow', 'red'] # Low K (Blue) -> High K (Red)
}

# We visualize the Log10 of the image to see the contrast

Map = geemap.Map(center=[39.5, -111.5], zoom=7)
Map.addLayer(dem, {'min': 4000, 'max': 11000, 'palette':get_palette('dem')}, 'Elevation')
# Map.addLayer(geomaterials, vis_params={'band':'palette':rock_palette}, name='GeoMaterials')
# Get unique classes to create an order
classes = geomaterials.aggregate_array('GeoMateria').distinct().sort()

def add_style_from_string(feature):
    # Find where this feature's class name sits in the list (0, 1, 2...)
    class_index = classes.indexOf(feature.get('GeoMateria'))
    color = ee.List(rock_palette).get(class_index)
    return feature.set('style', {
        'fillColor': color,   # The fill color (dynamic)
        'color': color,       # The outline color (dynamic)
        'width': 1            # Outline width (static, or make dynamic too)
    })

colored_geomaterials = geomaterials.map(add_style_from_string)

# 3. Visualize using .style()
# 'fillColor' tells GEE to look for the property name 'style_color' inside the feature
vector_viz = colored_geomaterials.style(
    styleProperty='style'                   # Outline width
)

# 4. Add to Map
# Note: vector_viz is now an RGB Image, so we don't need visualization params in addLayer
Map.addLayer(geomaterials, {}, 'GeoMaterials (Vectors)')
Map.addLayer(vector_viz, {}, 'GeoMaterials (Styled)')
# Map.addLayer(geomaterials, {}, 'GeoMaterials (Vector Style)')
# Map.addLayer(k_raster.log10(), vis_k, 'Hydraulic Conductivity (Log Scale)')
# Map.addLayer(k_raster, {
#     'min': 1e-8*m_s_to_m_month_conversion, 
#     'max': 1e-4*m_s_to_m_month_conversion, 
#     'palette': ['blue', 'cyan', 'green', 'yellow', 'red'] # Low K (Blue) -> High K (Red)
# }, 'Hydraulic Conductivity (m/month)')

# Map.addLayer(k_raster, {
#     'min': 0, 
#     'max': 1, 
#     'palette': ['blue', 'cyan', 'green', 'yellow', 'red'] # Low K (Blue) -> High K (Red)
# }, 'Hydraulic Conductivity (m/month)')

Map.addLayer(geok_raster_scaled.log10(), {
    'min': 0, 
    'max': 2, 
    'palette': ['blue', 'cyan', 'green', 'yellow', 'red'] # Low K (Blue) -> High K (Red)
}, 'Hydraulic Conductivity (m/month)')
# Map.addLayer(ksat, {
#     'min': 1e5, 
#     'max': 1e8, 
#     'palette': ['blue', 'cyan', 'green', 'yellow', 'red'] # Low K (Blue) -> High K (Red)
# }, 'ksat (m/month)')

# Map.addLayer(geok_difference, vis_params={'min':-10, 'max':10, 'palette':['red','white','blue']}, name='GeoK Difference')

Map.addLayer(dirty_devil_roi, {'color': 'red'}, 'Dirty Devil Gage Drainage Area')
Map.addLayer(duschesne_roi, {'color': 'blue'}, 'Duschesne Gage Drainage Area')
Map.addLayer(price_roi, {'color': 'green'}, 'Price River Gage Drainage Area')
Map.addLayer(sevier_roi, {'color': 'yellow'}, 'Sevier River Gage Drainage Area')
Map.addLayer(uinta_roi, {'color': 'purple'}, 'Uinta River Gage Drainage Area')
Map.addLayer(weber_roi, {'color': 'orange'}, 'Weber River Oakley Gage Drainage Area')




Map

Map(center=[39.5, -111.5], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright'…